* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/data_preview

In [1]:
import pandas as pd


def summarize_columns(df):
    print(
        pd.DataFrame(
            [
                (
                    c,
                    df[c].dtype,
                    len(df[c].unique()),
                    df[c].memory_usage(deep=True) // (1024**2),
                )
                for c in df.columns
            ],
            columns=["name", "dtype", "unique", "size (MB)"],
        )
    )
    print("Total size:", df.memory_usage(deep=True).sum() / 1024**2, "MB")
    print(f"df.shape:' {df.shape}")


df = pd.read_csv(
    "datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv", delimiter=","
)

summarize_columns(df)

/tmp/ipykernel_14878/719224584.py:23: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


                             name    dtype   unique  size (MB)
0                      CRASH DATE      str     5025         38
1                      CRASH TIME      str     1440         27
2                         BOROUGH      str        6         28
3                        ZIP CODE   object      439         77
4                        LATITUDE  float64   130328         17
5                       LONGITUDE  float64   101116         17
6                        LOCATION      str   393219         62
7                  ON STREET NAME      str    23176         65
8               CROSS STREET NAME      str    25238         46
9                 OFF STREET NAME      str   267508         30
10      NUMBER OF PERSONS INJURED  float64       33         17
11       NUMBER OF PERSONS KILLED  float64        8         17
12  NUMBER OF PEDESTRIANS INJURED    int64       14         17
13   NUMBER OF PEDESTRIANS KILLED    int64        6         17
14      NUMBER OF CYCLIST INJURED    int64        5    

## Discarding invalid rows & Redundant Columns

In [ ]:
# --- Removing Lat/Lon NaN values ---#

rows_before = df.shape[0]
df = df[df["LONGITUDE"].notna() & df["LATITUDE"].notna()]
print(f"-> Dropped {rows_before - df.shape[0]} rows after removing lat/lon NaNs rows")


# --- Removing redundant columns ---#

df = df.drop(["ZIP CODE", "LOCATION"], axis=1)


summarize_columns(df)

-> Dropped 240676 rows after removing lat/lon NaNs rows


                             name    dtype   unique  size (MB)
0                      CRASH DATE      str     5025         50
1                      CRASH TIME      str     1440         40
2                         BOROUGH      str        6         41
3                        LATITUDE  float64   130327         30
4                       LONGITUDE  float64   101115         30
5                  ON STREET NAME      str    16812         73
6               CROSS STREET NAME      str    19041         56
7                 OFF STREET NAME      str   250087         43
8       NUMBER OF PERSONS INJURED  float64       31         30
9        NUMBER OF PERSONS KILLED  float64        8         30
10  NUMBER OF PEDESTRIANS INJURED    int64       14         30
11   NUMBER OF PEDESTRIANS KILLED    int64        6         30
12      NUMBER OF CYCLIST INJURED    int64        5         30
13       NUMBER OF CYCLIST KILLED    int64        3         30
14     NUMBER OF MOTORIST INJURED    int64       29    

## Fixing missing BUROUGHS

In [5]:
import geopandas as gpd
from geodatasets import get_path

# -- Title Entries ex: BRONX -> Bronx
df["BOROUGH"] = df["BOROUGH"].str.title()


print(
    "Unique Boroughs include",
    list(df["BOROUGH"].unique()),
    "\nNumber of rows with missing 'BOROUGH' but present coordinates is:",
    sum(df["BOROUGH"].isna()),
)

missing_mask = df["BOROUGH"].isna()

missing = df.loc[missing_mask, ["LONGITUDE", "LATITUDE"]]

gdf_missing = gpd.GeoDataFrame(
    missing,
    geometry=gpd.points_from_xy(missing["LONGITUDE"], missing["LATITUDE"]),
    crs="EPSG:4326",
)

# --- Using geodatasets to source boroughs multipoligon ---#
path_to_boroughs = get_path("nybb")
boroughs = gpd.read_file(path_to_boroughs)

boroughs = boroughs.to_crs(
    "EPSG:4326"
)  # Convert to standard coordinate reference system


# --- Left spacial join gdf_missing with boroughs multi-poligon ---#
joined = gpd.sjoin(
    gdf_missing, boroughs[["BoroName", "geometry"]], how="left", predicate="within"
)

df.loc[missing_mask, "BOROUGH"] = joined["BoroName"]

print("Remaining missing BOROUGH after spatial fill:", df["BOROUGH"].isna().sum())

# --- Removing rows with BOROUGH == nan, after the fill ---#

df = df[df["BOROUGH"].notna()]
summarize_columns(df)

Unique Boroughs include ['Brooklyn', nan, 'Bronx', 'Manhattan', 'Queens', 'Staten Island'] 
Number of rows with missing 'BOROUGH' but present coordinates is: 484641
Remaining missing BOROUGH after spatial fill: 10094
                             name    dtype   unique  size (MB)
0                      CRASH DATE      str     5025         49
1                      CRASH TIME      str     1440         39
2                         BOROUGH      str        5         44
3                        LATITUDE  float64   130276         30
4                       LONGITUDE  float64   101069         30
5                  ON STREET NAME      str    16773         72
6               CROSS STREET NAME      str    19035         56
7                 OFF STREET NAME      str   249980         42
8       NUMBER OF PERSONS INJURED  float64       31         30
9        NUMBER OF PERSONS KILLED  float64        8         30
10  NUMBER OF PEDESTRIANS INJURED    int64       14         30
11   NUMBER OF PEDESTRIANS 

## Memory Optimization

In [6]:
df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"] + " " + df["CRASH TIME"], format="%m/%d/%Y %H:%M"
)

df = df.drop(["CRASH TIME", "CRASH DATE"], axis=1)


# --- Reducing memory footprint of numberic columns ---#
numeric_columns = [ #Small numeric values
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED",
]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int8")


# --- Reducing memory footprint of categorical columns ---#
category_columns = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "CONTRIBUTING FACTOR VEHICLE 3",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5",
    "VEHICLE TYPE CODE 1",
    "VEHICLE TYPE CODE 2",
    "VEHICLE TYPE CODE 3",
    "VEHICLE TYPE CODE 4",
    "VEHICLE TYPE CODE 5",
]

df = df.astype({k: "category" for k in category_columns})


# --- Reorder columns --- #
df = df[
    ["COLLISION_ID", "BOROUGH", "LATITUDE", "LONGITUDE"]
    + numeric_columns
    + category_columns
    + ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]
]

summarize_columns(df)
print(df.columns)

                             name     dtype   unique  size (MB)
0                    COLLISION_ID     int64  2002422         30
1                         BOROUGH       str        5         44
2                        LATITUDE   float64   130276         30
3                       LONGITUDE   float64   101069         30
4       NUMBER OF PERSONS INJURED      Int8       31         19
5        NUMBER OF PERSONS KILLED      Int8        8         19
6   NUMBER OF PEDESTRIANS INJURED      Int8       14         19
7    NUMBER OF PEDESTRIANS KILLED      Int8        6         19
8       NUMBER OF CYCLIST INJURED      Int8        5         19
9        NUMBER OF CYCLIST KILLED      Int8        3         19
10     NUMBER OF MOTORIST INJURED      Int8       29         19
11      NUMBER OF MOTORIST KILLED      Int8        6         19
12  CONTRIBUTING FACTOR VEHICLE 1  category       62         17
13  CONTRIBUTING FACTOR VEHICLE 2  category       62         17
14  CONTRIBUTING FACTOR VEHICLE 3  categ

In [7]:
df = df.reset_index(drop= True)
df.to_parquet("datasets/crash_data.parquet", engine= "pyarrow")

In [10]:
pd.Series(df["COLLISION_ID"].unique()).to_csv("datasets/collision_ids.csv", index= False, header= ["COLLISION_ID"])

In [12]:
df.columns

Index(['COLLISION_ID', 'BOROUGH', 'LATITUDE', 'LONGITUDE',
       'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
       'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
       'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
       'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED',
       'CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING FACTOR VEHICLE 2',
       'CONTRIBUTING FACTOR VEHICLE 3', 'CONTRIBUTING FACTOR VEHICLE 4',
       'CONTRIBUTING FACTOR VEHICLE 5', 'VEHICLE TYPE CODE 1',
       'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4',
       'VEHICLE TYPE CODE 5', 'ON STREET NAME', 'CROSS STREET NAME',
       'OFF STREET NAME'],
      dtype='str')